# Records, enums e igualdade

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Modelar dados simples com um record, e um conjunto fixo de valores com um enum.
- Diferenciar == de equals, e escrever equals e hashCode para que objetos funcionem como chaves em sets e maps.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Records: classes para dados simples
2. Conferindo os valores de um record, e mudando uma cópia
3. Enums: um conjunto fixo de valores
4. Enums com campos e métodos
5. == ou equals?
6. equals e hashCode numa classe

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Records: classes para dados simples

Muitas classes só guardam dados. Um **record** escreve todo o código de costume para você a partir de uma linha: `record Song(String title, int seconds) {}` dá um construtor, os acessores `title()` e `seconds()`, um `toString` legível, e um `equals` e um `hashCode` que comparam os valores. Os campos dele não mudam depois da criação. Ainda dá para acrescentar métodos dentro das chaves.

In [ ]:
record Song(String title, int seconds) {
    String length() {
        return seconds / 60 + ":" + String.format("%02d", seconds % 60);
    }
}

void main() {
    Song intro = new Song("Intro", 95);
    IO.println(intro);
    IO.println(intro.title() + " dura " + intro.length());
}

🧪 **Experimente:** acrescente um campo `String artist` ao record e conserte a chamada `new Song(...)`. Depois tente `intro.seconds = 10;`: records não mudam.

### ✏️ Exercício: A etiqueta de um objeto

Um Seat tem uma letra de fila e um número. Devolva a etiqueta dele, como `A1`, a partir dos acessores do record.

In [ ]:
record Seat(char row, int number) {}

String label(Seat seat) {
    // TODO: junte seat.row() e seat.number()
    return "";
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}

<details>
<summary>Ver uma resposta</summary>

```java
record Seat(char row, int number) {}

String label(Seat seat) {
    return "" + seat.row() + seat.number();
}

void main() {
    check("A1", label(new Seat('A', 1)), "A1");
    check("C12", label(new Seat('C', 12)), "C12");
}
```

</details>

## 2. Conferindo os valores de um record, e mudando uma cópia

Um record pode conferir os seus valores num **construtor compacto**: o nome do record sem parênteses, que roda antes de os campos receberem os valores. Como um record nunca muda, "mudar" um quer dizer criar um record novo com um valor trocado, normalmente num método chamado `withAlgumaCoisa`.

In [ ]:
record Song(String title, int seconds) {
    Song {                                        // o construtor compacto
        if (seconds <= 0) {
            throw new IllegalArgumentException("uma música precisa de duração positiva");
        }
    }

    Song withSeconds(int newSeconds) {
        return new Song(title, newSeconds);       // um record novo; este continua igual
    }
}

void main() {
    Song a = new Song("Intro", 95);
    Song b = a.withSeconds(120);
    IO.println(a + " / " + b);
    try {
        new Song("Silêncio", 0);
    } catch (IllegalArgumentException e) {
        IO.println("Recusado: " + e.getMessage());
    }
}

🧪 **Experimente:** acrescente um método `withTitle`. Depois recuse também um título vazio no construtor compacto.

## 3. Enums: um conjunto fixo de valores

Quando um valor só pode ser uma entre poucas opções (os dias da semana, os tamanhos de um copo), um **enum** as lista. Aí o compilador recusa qualquer outra coisa, `values()` devolve todas em ordem, `valueOf("SMALL")` transforma um texto numa delas, e um `switch` pode cobrir cada opção sem `default`.

In [ ]:
enum Direction { NORTH, EAST, SOUTH, WEST }

Direction turnRight(Direction d) {
    return switch (d) {
        case NORTH -> Direction.EAST;
        case EAST -> Direction.SOUTH;
        case SOUTH -> Direction.WEST;
        case WEST -> Direction.NORTH;
    };
}

void main() {
    Direction facing = Direction.valueOf("NORTH");
    facing = turnRight(facing);
    IO.println("Agora virado para " + facing + " (número " + facing.ordinal() + ")");
    for (Direction d : Direction.values()) {
        IO.print(d + " ");
    }
    IO.println();
}

🧪 **Experimente:** apague um `case` e leia o erro. Depois tente `Direction.valueOf("UP")` e leia a exceção.

### ✏️ Exercício: Tamanhos de copo

Devolva quantos mililitros cada tamanho tem: SMALL 250, MEDIUM 350, LARGE 500. Use um switch no enum.

In [ ]:
enum Size { SMALL, MEDIUM, LARGE }

int ml(Size size) {
    // TODO: um switch com um case por tamanho
    return 0;
}

void main() {
    check("pequeno", ml(Size.SMALL), 250);
    check("médio", ml(Size.MEDIUM), 350);
    check("grande", ml(Size.LARGE), 500);
}

<details>
<summary>Ver uma resposta</summary>

```java
enum Size { SMALL, MEDIUM, LARGE }

int ml(Size size) {
    return switch (size) {
        case SMALL -> 250;
        case MEDIUM -> 350;
        case LARGE -> 500;
    };
}

void main() {
    check("pequeno", ml(Size.SMALL), 250);
    check("médio", ml(Size.MEDIUM), 350);
    check("grande", ml(Size.LARGE), 500);
}
```

</details>

## 4. Enums com campos e métodos

Cada valor de um enum é um objeto, então um enum pode ter campos, um construtor e métodos, como uma classe. Os valores passam os seus dados para o construtor entre parênteses. Isso deixa os dados de cada valor junto do nome dele, em vez de num switch separado.

In [ ]:
enum Planet {
    MERCURY(0.38), EARTH(1.0), JUPITER(2.53);

    private final double gravity;     // em relação à Terra

    Planet(double gravity) {
        this.gravity = gravity;
    }

    double weigh(double kgOnEarth) {
        return kgOnEarth * gravity;
    }
}

void main() {
    for (Planet p : Planet.values()) {
        IO.println(p + ": " + String.format("%.1f", p.weigh(60)) + " kg");
    }
}

🧪 **Experimente:** acrescente MARS com gravidade 0.38. Depois acrescente um campo com a posição de cada planeta a partir do Sol, e imprima.

## 5. == ou equals?

Para objetos, o `==` pergunta "**o mesmo objeto**?": dois `new` criam dois objetos, mesmo com valores iguais. O `equals` pergunta "**os mesmos valores**?", mas só se a classe disser como compará-los. Records e Strings já dizem; para enums, o `==` serve, porque cada valor existe uma vez só. Os slides mostram isso na memória.

#### 🎞️ Objetos, referências e igualdade

![Dois news criam dois objetos mesmo com valores iguais; atribuir compartilha um objeto; == compara objetos e equals compara valores](objects-memory-steps.pt-br.svg)

In [ ]:
record Seat(char row, int number) {}

void main() {
    Seat s1 = new Seat('A', 1);
    Seat s2 = new Seat('A', 1);
    Seat s3 = s1;
    IO.println(s1 == s2);
    IO.println(s1.equals(s2));
    IO.println(s1 == s3);
}

🧪 **Experimente:** preveja as três linhas antes de rodar. Depois troque o `record Seat` por uma classe comum com os mesmos campos e um construtor: o que `s1.equals(s2)` dá agora?

## 6. equals e hashCode numa classe

Uma classe comum herda um `equals` que só compara referências. Para comparar valores, sobrescreva `equals(Object other)`: confira que `other` é o mesmo tipo de objeto, depois compare os campos. Sobrescreva **também** o `hashCode`, com `Objects.hash(...)` dos mesmos campos: um `HashSet` ou um `HashMap` olha primeiro o hash, e dois objetos iguais com hashes diferentes são tratados como diferentes.

In [ ]:
class Money {
    private final long cents;
    private final String currency;

    Money(long cents, String currency) {
        this.cents = cents;
        this.currency = currency;
    }

    @Override
    public boolean equals(Object other) {
        return other instanceof Money m && cents == m.cents && currency.equals(m.currency);
    }

    @Override
    public int hashCode() {
        return Objects.hash(cents, currency);
    }
}

void main() {
    Money a = new Money(500, "BRL");
    Money b = new Money(500, "BRL");
    IO.println(a.equals(b));
    Set<Money> wallet = new HashSet<>(List.of(a, b));
    IO.println(wallet.size());
}

🧪 **Experimente:** apague o método `hashCode` e rode: o `equals` continua dizendo true, mas o set pode guardar os dois. Apague também o `equals` e compare.

## Em resumo

Um **record** é uma classe para dados simples numa linha: `record Song(String title, int seconds) {}` escreve o construtor, os acessores `title()` e `seconds()`, o `toString`, o `equals` e o `hashCode`, e os campos dele nunca mudam. Um **enum** lista um conjunto fixo de valores, como `enum Size { SMALL, MEDIUM, LARGE }`.

```java
record Song(String title, int seconds) {}

enum Size { SMALL, MEDIUM, LARGE }

void main() {
    Song a = new Song("Intro", 95);
    Song b = new Song("Intro", 95);
    IO.println(a == b);          // false: dois objetos
    IO.println(a.equals(b));     // true: os mesmos valores
    Set<Song> playlist = new HashSet<>(List.of(a, b));
    IO.println(playlist.size()); // 1: músicas iguais contam uma vez

Size s = Size.valueOf("MEDIUM");
    IO.println(s + " é o tamanho número " + (s.ordinal() + 1));
}
```

`==` pergunta "o mesmo objeto?"; `equals` pergunta "os mesmos valores?". Uma classe comum herda um `equals` que só compara referências, então para comparar valores você sobrescreve o `equals` **e** o `hashCode` juntos.

**Cuidado:** dois objetos que são `equals` precisam ter o mesmo `hashCode`, senão um `HashSet` ou um `HashMap` os trata como diferentes.

## Leituras

- [Object as a Superclass](https://dev.java/learn/language/oop/inheritance/objects/) (dev.java (Oracle), 8 min): As seções sobre toString, equals e hashCode: por que o equals precisa ser sobrescrito para comparar valores, e por que o hashCode precisa mudar junto. Pode pular clone e finalize.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Pontos como records** · Fácil
- **Moedas (enum)** · Médio
- **Pontos iguais (equals e hashCode)** · Difícil
- Teste: **Sistema da biblioteca** · Difícil